In [1]:
x,y=var('x y')

In [2]:
poly1=x**2*y+y+4; poly2 = x*y**2+5

In [3]:
show(poly1);show(poly2)

x^2*y + y + 4

x*y^2 + 5

In [4]:
from functools import reduce
from operator import mul

def term_to_differential(term):
    pass

def to_differentials(p):
    from sage.symbolic.expression_conversions import polynomial
    vars = p.variables()
    q = polynomial(p, PolynomialRing(QQ, names = vars))
    big_result = []
    terms = list(q.iterator_exp_coeff())
    F = function('F')(*vars)
    for term in terms:
        dd = []        
        for idx, v in enumerate(term[0]):
            gurri_x = [p.variables()[idx] , v]
            dd.extend(gurri_x)
        big_result.append(term[1]*diff(F, *dd))
    return sum(big_result)

In [5]:

d1=to_differentials(poly1)
d2=to_differentials(poly2)

In [6]:
from operator import mul
def to_polynomial(d):    
    if d.operator().__name__ != "add_vararg":
        ops = [d]
    else:
        ops = d.operands()
    result = []
    #import pdb; pdb.set_trace()
    for o in ops:
        if "FUNCTION" in o.operator().__class__.__name__.upper():
            result.append(o.operands()[-1])        
        elif hasattr(o.operator(), "__name__") and o.operator().__name__=="mul_vararg":
            result.append(o.operands()[-1])
        else:
            variables=[0]*len(o.operands())
            for i in o.operator().parameter_set():
                variables[i] += 1  # XXX Use Count?
            local_result = \
            reduce(mul, 
                   ((_[0]^_[1]) for _ in zip(o.operands(), variables)), 1)     
            result.append(local_result)
            
    return sum(result)

In [7]:
d3 = d2+4*F(x,y)
to_polynomial(d3)

x*y^2 + 9

In [8]:
to_polynomial(d1)

x^2*y + y + 4

In [9]:
to_polynomial(d2)

x*y^2 + 5

In [10]:
bool(poly1==to_polynomial(to_differentials(poly1)))

True

In [11]:
bool(poly2==to_polynomial(to_differentials(poly2)))

True

In [12]:
to_polynomial(to_differentials(poly2))

x*y^2 + 5

In [13]:
poly2

x*y^2 + 5

In [14]:
to_differentials(poly2)

5*F(x, y) + diff(F(x, y), x, y, y)